# Preprocess Normbank

Generated from `src/preprocess/preprocess_normbank.py`.
The first code cell recreates script-like path behavior for notebook execution.


In [ ]:
from pathlib import Path
import sys

_NOTEBOOK_BOOTSTRAP_VERBOSE = True

if _NOTEBOOK_BOOTSTRAP_VERBOSE:
    print("[bootstrap] cwd =", Path.cwd().resolve())

if "ipykernel" in sys.modules:
    # Avoid argparse failures from Jupyter kernel launch flags.
    sys.argv = [sys.argv[0]]
    if _NOTEBOOK_BOOTSTRAP_VERBOSE:
        print("[bootstrap] detected ipykernel, trimmed sys.argv to:", sys.argv)

_SOURCE_RELATIVE_PATH = Path("src/preprocess/preprocess_normbank.py")
_repo_root = None
for _candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if _NOTEBOOK_BOOTSTRAP_VERBOSE:
        print("[bootstrap] checking candidate:", _candidate)
    if (_candidate / _SOURCE_RELATIVE_PATH).exists():
        _repo_root = _candidate
        if _NOTEBOOK_BOOTSTRAP_VERBOSE:
            print("[bootstrap] matched repo root:", _repo_root)
        break

if _repo_root is None:
    _repo_root = Path.cwd().resolve()
    if _NOTEBOOK_BOOTSTRAP_VERBOSE:
        print("[bootstrap] no match found, falling back to cwd:", _repo_root)

_source_file = (_repo_root / _SOURCE_RELATIVE_PATH).resolve()
__file__ = str(_source_file)
if _NOTEBOOK_BOOTSTRAP_VERBOSE:
    print("[bootstrap] source relative path =", _SOURCE_RELATIVE_PATH)
    print("[bootstrap] resolved __file__ =", __file__)

for _path in (str(_repo_root), str(_source_file.parent)):
    if _path not in sys.path:
        sys.path.insert(0, _path)
        if _NOTEBOOK_BOOTSTRAP_VERBOSE:
            print("[bootstrap] added to sys.path:", _path)
    elif _NOTEBOOK_BOOTSTRAP_VERBOSE:
        print("[bootstrap] already on sys.path:", _path)


In [ ]:
from pathlib import Path

import _common


def _from_normbank_csv(path: Path):
    rows = []
    for row in _common.read_csv_rows(path, errors="replace"):
        metadata = {
            key: row.get(key, "")
            for key in [
                "setting",
                "behavior",
                "constraints",
                "constraints_given",
                "constraint_predict",
            ]
            if row.get(key, "") not in (None, "")
        }
        rows.append(
            {
                "text": str(row.get("norm", "")).strip(),
                "label": str(row.get("label", "")).strip(),
                "dataset": "normbank",
                "task": "norm_classification",
                "split": str(row.get("split", "")).strip(),
                "source_file": path.name,
                "metadata": metadata,
            }
        )
    return rows


def _from_legacy_text_tree(base: Path):
    rows = []
    for path in sorted(base.rglob("*.txt")):
        rel = path.relative_to(base)
        category = rel.parts[0]
        subcategory = "/".join(rel.parts[1:-1]) if len(rel.parts) > 2 else ""
        file_name = rel.stem
        with path.open("r", encoding="utf-8", errors="replace") as f:
            for line in f:
                text = line.strip()
                if not text:
                    continue
                rows.append(
                    {
                        "text": text,
                        "category": category,
                        "subcategory": subcategory,
                        "file": file_name,
                        "source": "normbank",
                        "source_file": str(rel),
                    }
                )
    return rows


def main() -> None:
    csv_path = _common.RAW_ROOT / "normbank" / "NormBank.csv"
    legacy_base = _common.RAW_ROOT / "normbank-main" / "data" / "raw"

    if csv_path.exists():
        rows = _from_normbank_csv(csv_path)
    elif legacy_base.exists():
        rows = _from_legacy_text_tree(legacy_base)
    else:
        print(f"Missing NormBank raw data. Checked: {csv_path} and {legacy_base}")
        return

    out_dir = _common.PROCESSED_ROOT / "normbank"
    _common.write_jsonl(out_dir / "normbank.jsonl", rows)
    _common.write_csv(out_dir / "normbank.csv", rows)
    print(f"Wrote {len(rows)} records to {out_dir}")


## Entrypoint

Run the original script entrypoint when needed.


In [ ]:
main()
